# 20 · Query Transformation

**Idea:** the user's raw question is often a poor search query. Rewriting it before retrieval lifts recall. Three classic transforms:

- **HyDE** — hallucinate a hypothetical answer and retrieve with *that* (a doc that *looks like* the answer sits near the real answer in embedding space).
- **Step-back** — ask a broader background question so foundational context is retrieved too.
- **Multi-query / RAG-Fusion** — generate several paraphrases, retrieve each, and fuse the ranked lists with Reciprocal Rank Fusion.

> Transforms run on Ollama by default. One cell optionally uses OpenAI — skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Build a store and set a deliberately terse query — the kind that under-retrieves because it shares few words with the corpus.

In [ ]:
from rag_lab import load_markdown, NumpyStore

store = NumpyStore(provider='ollama'); store.add(load_markdown())
query = 'why rerank?'

print('baseline retrieval:')
for h in store.search(query, k=3):
    print(f'  {h.score:.3f}  [{h.chunk.source}]  {h.chunk.text[:80]}...')

**HyDE:** generate a plausible answer paragraph, then retrieve with the paragraph instead of the terse query. The richer text matches more relevant chunks.

In [ ]:
from rag_lab import hyde
from study_buddy import CostTracker

tracker = CostTracker()
hypo = hyde(query, provider='ollama', tracker=tracker)
print('hypothetical doc:\n', hypo[:300], '...\n')

print('HyDE retrieval:')
for h in store.search(hypo, k=3):
    print(f'  {h.score:.3f}  [{h.chunk.source}]  {h.chunk.text[:80]}...')
print('\ncost:', tracker.summary())

**Step-back:** widen the question to pull in background context alongside the specifics.

In [ ]:
from rag_lab import step_back

broad = step_back(query, provider='ollama')
print('step-back question:', broad)

**RAG-Fusion:** paraphrase the query several ways, retrieve each, and fuse with Reciprocal Rank Fusion — chunks that surface across multiple phrasings rise to the top.

In [ ]:
from rag_lab import rag_fusion, multi_query

print('variants:', multi_query(query, n=3, provider='ollama'), '\n')

fused = rag_fusion(query, store, n=3, k=4, provider='ollama')
print('fused top results:')
for h in fused:
    print(f'  {h.score:.4f}  [{h.chunk.source}]  {h.chunk.text[:80]}...')

Optionally run the paraphrase step on OpenAI. Skips if no key.

In [ ]:
try:
    variants = multi_query(query, n=3, provider='openai', model='gpt-4o-mini')
    print('openai variants:')
    for v in variants:
        print(' -', v)
except Exception as e:
    print('openai skipped:', type(e).__name__, e)

**Takeaway:** when retrieval misses, fix the *query* before blaming the index. HyDE and step-back reshape a single query; RAG-Fusion spreads risk across paraphrases and fuses with RRF (the same fusion used for hybrid search in notebook 10).

Next up → `21_rag_evaluation.ipynb`, where we finally *measure* whether any of this helps.